# FileMind — train locally-honest models on Google Colab

This notebook trains the three FileMind classifiers **from scratch** (no pretrained
weights, no external datasets) and exports the transformer to ONNX for the desktop app.

**Honesty rules**
- Every number printed here is measured, never estimated.
- Synthetic data inflates metrics — mark results accordingly in MODEL_CARD.md.
- To train on *your* files, build a manifest per `ml/data/README.md` and use `--from-scan`.


In [ ]:
# 1. Get the code
import sys, os
if not os.path.exists('filemind/ml/train_transformer_placeholder.py'):
    !git clone https://github.com/YOUR_USERNAME/filemind.git
    %cd filemind
%cd /content/filemind
!ls ml

In [ ]:
# 2. Generate the synthetic dataset (or upload your own manifest and use --from-scan)
!python ml/training/prepare_dataset.py --synthetic --per-class 2000 --out ml/data/dataset.jsonl

In [ ]:
# 3. Model A — TF-IDF + Logistic Regression baseline (the floor to beat)
!python ml/training/train_baseline.py --data ml/data/dataset.jsonl --out ml/artifacts/baseline.joblib
!python ml/evaluation/evaluate.py --baseline ml/artifacts/baseline.joblib --data ml/data/dataset.jsonl --out ml/artifacts/evaluation-report.json

In [ ]:
# 4. Model B — TextCNN (from scratch)
!python ml/training/train_textcnn.py --data ml/data/dataset.jsonl --out ml/artifacts/textcnn.pt --epochs 8

In [ ]:
# 5. Model C — FileTransformer (from scratch, ~3M params)
#    GPU recommended; ~1 min/epoch on T4 for 26k samples
!python ml/training/train_transformer.py --data ml/data/dataset.jsonl \
    --out ml/artifacts/filemind-transformer.pt --epochs 10

In [ ]:
# 6. Real evaluation — accuracy, macro-F1, per-class, confusion matrix
!python ml/evaluation/evaluate.py --torch ml/artifacts/filemind-transformer.pt --arch transformer \
    --data ml/data/dataset.jsonl --out ml/artifacts/evaluation-report.json
import json
print(json.dumps(json.load(open('ml/artifacts/evaluation-report.json')), indent=2)[:1200])

In [ ]:
# 7. Export ONNX for the desktop app
!python ml/export/export_onnx.py --checkpoint ml/artifacts/filemind-transformer.pt \
    --arch transformer --model-dir models
# sanity: parity between PyTorch and ONNX
!python - <<'EOF'
import numpy as np, onnxruntime as ort, sys, torch, json
sys.path.insert(0, 'ml')
from filemind_ml.tokenizer import WordVocab
from filemind_ml.models.transformer import FileTransformer
sess = ort.InferenceSession('models/filemind-transformer.onnx')
ckpt = torch.load('ml/artifacts/filemind-transformer.pt', map_location='cpu')
vocab = WordVocab.load('models/vocab.json')
m = FileTransformer(vocab_size=ckpt['vocab_size'], **ckpt['config']); m.load_state_dict(ckpt['state_dict']); m.eval()
ids = torch.tensor([vocab.encode('invoice_2026_03_acme.pdf invoice total due')], dtype=torch.long)
with torch.no_grad(): pt = m(ids, torch.ones((1,64), dtype=torch.float32)).numpy()
ox = sess.run(None, {'input_ids': ids.numpy(), 'attention_mask': np.ones((1,64), dtype=np.float32)})[0]
print('max |pt-onnx| =', float(np.abs(pt-ox).max()))
EOF

In [ ]:
# 8. Download the artifacts and drop them into your repo's models/ folder
from google.colab import files
files.download('models/filemind-transformer.onnx')
files.download('models/vocab.json')
files.download('models/labels.json')
files.download('models/manifest.json')
files.download('ml/artifacts/evaluation-report.json')

## 9. Update MODEL_CARD.md — with the numbers you actually got

Copy the printed test accuracy / macro-F1 / parameter counts into the results
table, and note the dataset used (`synthetic-v1`, per-class size, date).
If you trained on your own files, say so — that's the number worth quoting.
